# Make shapefile grid

To open, use "env-shapefiles":
```
- conda create --name env-shapefiles python=3.10.10
- conda activate env-shapefiles
- conda install numpy scipy matplotlib xarray scikit-learn datashader jupyterlab palettable seaborn
- pip install rasterio
- conda install -c conda-forge dask netCDF4 regionmask
- conda install -c conda-forge cartopy geopandas
```

In [1]:
import os
import geopandas as gpd
import pandas as pd
import rasterio
import regionmask
import numpy as np
import matplotlib.pyplot as plt
import predictions
import importlib as imp

from rasterio.transform import Affine

In [2]:
mosaic_filename = "/Users/eabarnes/Documents/shapefile_mosaic.tif"
regs_shp = gpd.read_file("data/shapefiles/ne_10m_admin_0_countries.shp")


In [ ]:
with rasterio.open("data/hii_coastal_buffer_mask.tif") as buffer_mask:
    __, lats = buffer_mask.xy(range(buffer_mask.height), 0, offset="ul")
    lons, __ = buffer_mask.xy(0, range(buffer_mask.width), offset="ul")
    meta = buffer_mask.meta
print(np.min(lons), np.max(lons))
print(np.min(lats), np.max(lats))

print(np.shape(lats), np.shape(lons))

In [ ]:
N, M = 4, 4
chunk_lon = int(len(lons) / M)
chunk_lat = int(len(lats) / N)
print(f"{chunk_lat = }, {chunk_lon = }\n")
res_lon = np.diff(lons)[0]
res_lat = np.diff(lats)[0]

NODATA = 999
filenames_list = []

for group_lon in np.arange(0, M):
    for group_lat in np.arange(0, N):
        print(f"{group_lon = }, {group_lat = }")

        mask_country = 0.0

        start, end = group_lon * chunk_lon, (group_lon + 1) * chunk_lon
        if group_lon == M - 1:
            end = len(lons)
        sub_lons = lons[start:end]

        start, end = group_lat * chunk_lat, (group_lat + 1) * chunk_lat
        if group_lat == N - 1:
            end = len(lats)
        sub_lats = lats[start:end]

        print(len(sub_lats), len(sub_lons))

        # check if file exists
        filename = "/Users/eabarnes/Documents/country_mask_" + str(group_lon) + "_" + str(group_lat) + ".tif"
        if os.path.exists(filename):
            filenames_list.append(filename)
            continue

        print("...runnning mask_geopandas")
        mask_country = regionmask.mask_geopandas(regs_shp, sub_lons, sub_lats)

        print("...replacing missing values")
        mask_country = np.where(np.isnan(mask_country), NODATA, mask_country).astype("uint16")
        print(np.shape(mask_country))

        meta_data = meta.copy()
        meta_data["nodata"] = NODATA
        meta_data["width"] = np.shape(mask_country)[1]
        meta_data["height"] = np.shape(mask_country)[0]
        meta_data["dtype"] = mask_country.dtype
        meta_data["compress"] = "lzw"
        meta_data["transform"] = Affine.translation(np.min(sub_lons), np.max(sub_lats)) * Affine.scale(res_lon, res_lat)

        print("...saving the tif file")
        with rasterio.open(filename, "w", **meta_data) as dst:
            dst.write(mask_country, 1)
            dst.set_band_description(1, "country mask")

        filenames_list.append(filename)

# put all of the files together
mosaic, mosaic_trans = predictions.create_mosaic(filenames_list)
meta_data = predictions.save_predictions_tif(mosaic, mosaic_filename, trans=mosaic_trans, nodata=NODATA)
print("mosaic saved.")


In [5]:
r = pd.DataFrame(regs_shp.drop(columns="geometry"))
r.to_pickle("/Users/eabarnes/Documents/shapefile_dataframe.pkl")

In [ ]:
with rasterio.open(mosaic_filename) as mf:
    print(mf.meta)